# Titanic Survival Prediction 🚢

## Project Overview
In this project, we analyze the Titanic dataset and build a machine learning model to predict passenger survival.

Main steps:
- Exploratory Data Analysis (EDA)
- Handling missing values
- Categorical and numerical feature analysis
- Feature engineering
- Building a Logistic Regression pipeline
- Model evaluation and error analysis
- Interpreting model coefficients


In [1]:
import seaborn as sns
import pandas as pd
import numpy as np

df = sns.load_dataset("titanic")

df.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 1. Initial Data Exploration

In [2]:
df.info()

df.isnull().sum()


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

## 2. Exploring Relationships With Survival

In [3]:
# Compare numerical features between survivors and non-survivors

df.groupby("survived")[["age", "fare"]].mean()


,age,fare
survived,,
0,30.626179,22.117887
1,28.343690,48.395408


In [4]:
# Survival rate by gender

df.groupby("sex")["survived"].mean()


sex
female    0.742038
male      0.188908
Name: survived, dtype: float64

In [5]:
# Boolean features can be converted to integers for correlation analysis

df[["survived", "age", "fare", "sibsp", "parch", "adult_male", "alone"]].corr()


,survived,age,fare,sibsp,parch,adult_male,alone
survived,1.000000,-0.077221,0.257307,-0.035322,0.081629,-0.557080,-0.203367
age,-0.077221,1.000000,0.096067,-0.308247,-0.189119,0.280328,0.198270
fare,0.257307,0.096067,1.000000,0.159651,0.216225,-0.182024,-0.271832
sibsp,-0.035322,-0.308247,0.159651,1.000000,0.414838,-0.253586,-0.584471
parch,0.081629,-0.189119,0.216225,0.414838,1.000000,-0.349943,-0.583398
adult_male,-0.557080,0.280328,-0.182024,-0.253586,-0.349943,1.000000,0.404744
alone,-0.203367,0.198270,-0.271832,-0.584471,-0.583398,0.404744,1.000000


In [6]:
# Survival rate by passenger class

df.groupby("class", observed=True)["survived"].mean()


class
First     0.629630
Second    0.472826
Third     0.242363
Name: survived, dtype: float64

## 3. Categorical Feature Relationships

In [7]:
# Relationship between gender and survival

pd.crosstab(df["sex"], df["survived"])


survived,0,1
sex,,
female,81,233
male,468,109


In [8]:
# Measuring association between categorical variables using Cramer's V

from scipy.stats.contingency import association

table = pd.crosstab(df["sex"], df["survived"])
association(table.to_numpy(), method="cramer")


0.5433513806577551

## 4. Feature Engineering

In [9]:
# Combine gender and passenger class

df["sex_class"] = (
    df["sex"].astype(str) + "_" + df["class"].astype(str)
)

df.groupby("sex_class", observed=True)["survived"].agg(
    ["count", "mean"]
).sort_values("mean", ascending=False)


,count,mean
sex_class,,
female_First,94,0.968085
female_Second,76,0.921053
female_Third,144,0.500000
male_First,122,0.368852
male_Second,108,0.157407
male_Third,347,0.135447


In [10]:
# Create family size feature

df["family_size"] = df["sibsp"] + df["parch"] + 1

df[["sibsp", "parch", "family_size"]].head()


,sibsp,parch,family_size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


In [11]:
# Convert family size into meaningful groups

df["family_group"] = pd.cut(
    df["family_size"],
    bins=[0, 1, 4, float("inf")],
    labels=["alone", "small", "large"]
)

df.groupby("family_group", observed=True)["survived"].agg(
    ["mean", "count"]
)


,mean,count
family_group,,
alone,0.303538,537
small,0.578767,292
large,0.161290,62


In [12]:
# Interaction between family group and gender

df["sex_family"] = (
    df["family_group"].astype(str) + "_" + df["sex"].astype(str)
)

df.groupby(["sex", "family_group"], observed=True)["survived"].agg(
    ["mean", "count"]
)


mean  count
sex    family_group                 
female alone         0.785714    126
       small         0.806452    155
       large         0.272727     33
male   alone         0.155718    411
       small         0.321168    137
       large         0.034483     29

## 5. Machine Learning Pipeline

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

numeric_features = ["age", "fare"]
categorical_base = ["sex", "class", "family_group"]
categorical_new = categorical_base + ["sex_family"]

X = df[numeric_features + categorical_new]
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [14]:
def make_model(categorical_features):
    # Preprocess numerical and categorical features separately
    preprocess = ColumnTransformer([
        ("num", make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler()
        ), numeric_features),

        ("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore")
        ), categorical_features)
    ])

    # Combine preprocessing and Logistic Regression
    return make_pipeline(
        preprocess,
        LogisticRegression(max_iter=1000)
    )


model_a = make_model(categorical_base)
model_b = make_model(categorical_new)


In [15]:
model_a.fit(X_train, y_train)
model_b.fit(X_train, y_train)

pred_a = model_a.predict(X_test)
pred_b = model_b.predict(X_test)

print("Model A:", accuracy_score(y_test, pred_a))
print("Model B:", accuracy_score(y_test, pred_b))


Model A: 0.8212290502793296
Model B: 0.8156424581005587


In [16]:
print(X_train.columns.tolist())
print(X_train.head())

['age', 'fare', 'sex', 'class', 'family_group', 'sex_family']
      age      fare     sex   class family_group    sex_family
692   NaN   56.4958    male   Third        alone    alone_male
481   NaN    0.0000    male  Second        alone    alone_male
527   NaN  221.7792    male   First        alone    alone_male
855  18.0    9.3500  female   Third        small  small_female
801  31.0   26.2500  female  Second        small  small_female


## 6. Model Evaluation

In [17]:
print("=== Model A ===")
print(classification_report(y_test, pred_a))
print(confusion_matrix(y_test, pred_a))

print("=== Model B ===")
print(classification_report(y_test, pred_b))
print(confusion_matrix(y_test, pred_b))


=== Model A ===
              precision    recall  f1-score   support

           0       0.83      0.89      0.86       110
           1       0.80      0.71      0.75        69

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179

[[98 12]
 [20 49]]
=== Model B ===
              precision    recall  f1-score   support

           0       0.83      0.88      0.85       110
           1       0.79      0.71      0.75        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.82      0.81       179

[[97 13]
 [20 49]]


## 7. Error Analysis

In [18]:
errors = X_test.copy()

errors["actual"] = y_test
errors["prediction"] = pred_a

errors[errors["actual"] != errors["prediction"]].groupby(
    ["sex", "class", "actual", "prediction"]
).size().sort_values(ascending=False)


sex     class   actual  prediction
male    Third   1       0             10
female  Third   0       1              9
male    First   1       0              7
        Second  1       0              3
female  First   0       1              1
        Second  0       1              1
male    First   0       1              1
dtype: int64

## 8. Logistic Regression Coefficient Analysis

In [19]:
preprocessor = model_a.named_steps["columntransformer"]

feature_names = preprocessor.get_feature_names_out()

coefficients = model_a.named_steps["logisticregression"].coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

importance.sort_values(
    "coefficient",
    ascending=False
)


,feature,coefficient
2,cat__sex_female,1.319843
4,cat__class_First,0.989111
9,cat__family_group_small,0.735904
7,cat__family_group_alone,0.488478
1,num__fare,0.140447
5,cat__class_Second,-0.040039
0,num__age,-0.450237
6,cat__class_Third,-0.943321
8,cat__family_group_large,-1.218631
3,cat__sex_male,-1.314092
